In [0]:
%run ./create_table_utillity

#### Reading plane data using autoloader

In [0]:
from pyspark.sql.functions import to_date, current_timestamp

# Reading data
df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", "/dbfs/FileStore/tables/schema/PLANE")
    .load("/mnt/geeks/rw_adls/PLANE/")
)


df = df.withColumn("Date_Part", to_date(current_timestamp()))

display(df)

In [0]:
# Writing data
df_base = df.selectExpr(
    "tailnum as tailid",
    "type",
    "manufacturer",
    "to_date(issue_date) as issue_date",
    "model",
    "status",
    "aircraft_type",
    "engine_type",
    "cast(year as int) as year",
    "to_date(Date_Part,'yyyy-MM-dd') as Date_Part",
)


display(df_base)

In [0]:
df_base.writeStream.trigger(once=True).format("delta").option(
    "checkpointLocation", "/dbfs/FileStore/tables/checkpointLocation/PLANE"
).start("/mnt/geeks/cld_adls/plane")

#### Creating delata table on the data

In [0]:
f_delta_cleansed_load('plane', "abfss://cleansed@geekadlsstoragesink.dfs.core.windows.net/plane/", 'cleansed_geekcoders')

In [0]:
%sql

select * from cleansed_geekcoders.plane;